## Structured Output

##### Models can be requested to provide their response in a certain format or a given schema.

# Pydantic

#### Pydantic models provide the richest feature set with field validation, description and nested structure.

In [26]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3.6-27b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x10eca1f40>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10ecc3830>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [27]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The release year of the movie")
    director: str = Field(description="The director of the movie")
    genre: str = Field(description="The genre of the movie")
    rating: float = Field(description="The rating of the movie on a scale of 1 to 10")

In [28]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x10eca1f40>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10ecc3830>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The release year of the movie', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'genre': {'description': 'The genre of the movie', 'type': 'string'}, 'rating': {'description': 'The rating of the movie on a scale of 1 to 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'genre', 'rating'], 'type': 'obje

In [29]:
model.invoke("Provide details of movie Dhoom 2")

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Request**: The user is asking for details about the movie "Dhoom 2". This is a straightforward request for information about a specific film.\n\n2.  **Identify Key Information Needed**: For a movie, typical details include:\n   - Title\n   - Release year\n   - Director\n   - Producers\n   - Screenplay/Story writers\n   - Cast (main actors/actresses)\n   - Music composer\n   - Genre\n   - Runtime\n   - Plot summary\n   - Box office performance/reception\n   - Sequel/Prequel context\n   - Notable facts/awards\n\n3.  **Retrieve Knowledge (Internal Training Data)**:\n   - *Title*: Dhoom 2\n   - *Year*: 2006\n   - *Director*: Sanjay Gupta\n   - *Producer*: Abbas-Mustan (Wait, no. Producers are Abbas-Mustan? Actually, Dhoom 2 was produced by Abbas-Mustan and Vikram Bhatt? Let me verify. Actually, the producers are Abbas-Mustan and Vikram Bhatt under the banner of Abbas-Mustan Productions and Reliance Big Enter

In [30]:
model_with_structure.invoke("Provide details of movie Dhoom 2")

Movie(title='Dhoom 2', year=2006, director='Sanjay Gadhvi', genre='Action', rating=6.0)

In [34]:
### Message output alongside parsed structured output

from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The release year of the movie")
    director: str = Field(description="The director of the movie")
    genre: str = Field(description="The genre of the movie")
    rating: float = Field(description="The rating of the movie on a scale of 1 to 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details of movie Dhoom 2")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:** The user is asking for details of the movie "Dhoom 2".\n2.  **Identify Required Information:** To use the `Movie` function, I need:\n   - title: "Dhoom 2"\n   - year: Need to find the release year\n   - director: Need to find the director\n   - genre: Need to find the genre\n   - rating: Need to find the rating (scale 1-10)\n3.  **Retrieve Knowledge (Internal):**\n   - Title: Dhoom 2\n   - Year: 2006\n   - Director: Sanjay Gadhvi\n   - Genre: Action, Crime, Thriller (typically classified as Action/Crime)\n   - Rating: IMDB rating is around 6.2/10. I\'ll use 6.2.\n4.  **Validate Parameters:**\n   - title: "Dhoom 2" (string)\n   - year: 2006 (integer)\n   - director: "Sanjay Gadhvi" (string)\n   - genre: "Action" (string) - I\'ll stick to a primary genre\n   - rating: 6.2 (number)\n   All required parameters are present and match the expected types.\n5.  **Con

# Nested Structure

In [41]:
### Message output alongside parsed structured output

from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genre: str = Field(description="The genre of the movie")
    budget: float | None = Field(description="The budget of the movie in USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details of movie Dhoom 2")
response

MovieDetails(title='Dhoom 2', year=2006, cast=[Actor(name='Aamir Khan', role='Samar Ghosh'), Actor(name='Abhishek Bachchan', role='Jai'), Actor(name='Uday Chopra', role='Ali'), Actor(name='Katrina Kaif', role='Nisha'), Actor(name='Elisha Cuthbert', role='Sandy'), Actor(name='Zayed Khan', role='Inspector Ali')], genre='Action, Crime, Thriller', budget=10000000.0)

# TypeDict

##### Provides a simpler alternative using built in typing, ideal when you don't need runtime validation.

In [44]:
from typing_extensions import Annotated, TypedDict

class MovieDict(TypedDict):
    title: Annotated[str, "The title of the movie"]
    year: Annotated[int, "The release year of the movie"]
    director: Annotated[str, "The director of the movie"]
    rating: Annotated[float, "The rating of the movie out of 10"]

model_with_typedict = model.with_structured_output(MovieDict)
response = model_with_typedict.invoke("Please provide details of movie Spiderman")
response

{'director': 'Sam Raimi', 'rating': 7.4, 'title': 'Spider-Man', 'year': 2002}

In [46]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genre: str = Field(description="The genre of the movie")
    budget: float | None = Field(description="The budget of the movie in USD")

model_with_typedict = model.with_structured_output(MovieDetails)

response = model_with_typedict.invoke("Provide details of movie Spiderman")
response

{'budget': 139000000,
 'cast': [{'name': 'Tobey Maguire', 'role': 'Peter Parker / Spider-Man'},
  {'name': 'Willem Dafoe', 'role': 'Norman Osborn / Green Goblin'},
  {'name': 'Kirsten Dunst', 'role': 'Mary Jane Watson'},
  {'name': 'James Franco', 'role': 'Harry Osborn'}],
 'genre': 'Action, Adventure, Sci-Fi',
 'title': 'Spider-Man',
 'year': 2002}

In [63]:
model.profile # SUpported for Qwen 3 model only

# DataClasses

#### A data class is a class typically containing mainly data, although there aren't really any restrictions. We create it using @dataclass decorator.

In [79]:
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [80]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact information from the following text: Dhruv Yadav, dhruvrao@example.com, (123) 456-7890"
    }]
})

result

{'messages': [HumanMessage(content='Extract contact information from the following text: Dhruv Yadav, dhruvrao@example.com, (123) 456-7890', additional_kwargs={}, response_metadata={}, id='bd4c9291-61cd-4a8a-8297-816c4661e842'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User wants me to extract contact information from the provided text.\n   - Text: "Dhruv Yadav, dhruvrao@example.com, (123) 456-7890"\n   - Available tool: `ContactInfo` with parameters `name`, `email`, `phone` (all required).\n\n2.  **Identify Entities in Text:**\n   - Name: Dhruv Yadav\n   - Email: dhruvrao@example.com\n   - Phone: (123) 456-7890\n\n3.  **Map to Tool Parameters:**\n   - `name`: "Dhruv Yadav"\n   - `email`: "dhruvrao@example.com"\n   - `phone`: "(123) 456-7890"\n\n4.  **Construct Tool Call:**\n   - Call `ContactInfo` with the extracted parameters.\n\n5.  **Execute/Output:**\n   - Generate the function call.✅\n   - No

In [82]:
result["structured_response"]

ContactInfo(name='Dhruv Yadav', email='dhruvrao@example.com', phone='(123) 456-7890')

In [1]:
## TypeDict

from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person"""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact information from the following text: Dhruv Yadav, dhruvrao@example.com, (123) 456-7890"
    }]
})

result["structured_response"]

{'name': 'Dhruv Yadav',
 'email': 'dhruvrao@example.com',
 'phone': '(123) 456-7890'}

In [2]:
## Data Class

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person"""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    response_format=ContactInfo
)

result= agent.invoke({
    "messages": [{
            "role": "user",
            "content": "Extract contact information from the following text: Dhruv Yadav, dhruvrao@example.com, (123) 456-7890"
        }]
})

result["structured_response"]

ContactInfo(name='Dhruv Yadav', email='dhruvrao@example.com', phone='(123) 456-7890')